# Modeling Objectives and Questions

In this modeling stage, we focus on **two main outcomes**:

            "metadata": {
                "language": "markdown"
            },
2. **Scientific understanding**: Which structural properties influence CO2 uptake most, and does their influence change with pressure?

## Core Modeling Question

> Using LCD, PLD, void fraction, and surface area, how accurately can we predict CO2 uptake at different pressures, and which features are most important at each pressure?

## What We Will Examine

- Whether model performance differs between **low-pressure** and **high-pressure** regimes.
- Whether **surface area** or **void fraction** becomes more influential at higher pressure.
- Whether **LCD** and **PLD** provide distinct predictive value despite their strong correlation.
- Whether **nonlinear models** outperform **linear models**, consistent with nonlinear patterns observed in EDA.

## Features Used

- **LCD** (Largest Cavity Diameter)
- **PLD** (Pore Limiting Diameter)
- **Void fraction**
- **Surface area**

## Target

- **CO2 uptake** at each pressure point
==============================================================

In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Load Data
data = pd.read_csv('/home/susan/mof-co2-adsorption/data/processed/data_clean_v2')
# Copy dataframe
df= data.copy()
print(df.columns)
print("="*30)


Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg'],
      dtype='object')


In [17]:
# Step 2 Ml processing 
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, PolynomialFeatures, StandardScaler, MinMaxScaler  # for l
from sklearn.utils import shuffle
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
#Cross-validation
from sklearn.model_selection import GridSearchCV, KFold

In [6]:
# Store the four structural features used in the relationship analysis.
feature_columns = ["lcd", "pld", "void_fraction", "surface_area_m2g"]

# Store the five CO2-uptake targets in increasing pressure order.
target = "CO2_uptake_0.01bar_molkg"

# Select only the required features and targets
features = df[feature_columns].copy()
target = df["CO2_uptake_0.01bar_molkg"].copy()
# split data
# temp simply means temporary.
# X_train: 60%
# X_temp: 40%

# Shuffle the data only when we create the train/validation/test split—before standardization and before training any model.
X_train, X_temp, y_train, y_temp = train_test_split(features , target, test_size = 0.4 ,
                                                        random_state = 12345, shuffle=True)
                                                      
                                                       

# Train: 60%
# Validation: 20%
# Test: 20%
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=12345,
    shuffle=True
)
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_valid.shape, y_valid.shape)
print("Testing:", X_test.shape, y_test.shape)

print("\nMissing values in training features:")
print(X_train.isna().sum())


Training: (18740, 4) (18740,)
Validation: (6247, 4) (6247,)
Testing: (6247, 4) (6247,)

Missing values in training features:
lcd                 0
pld                 0
void_fraction       0
surface_area_m2g    0
dtype: int64


In [7]:

# Create simple baseline model
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train)

dummy_predictions = dummy_model.predict(X_valid)
print(dummy_predictions[:2])

[0.12550654 0.12550654]


==============================================================

The Dummy Regressor was used as a baseline. It predicts the mean CO₂ uptake at each pressure without using the structural features. Therefore, all later models should outperform it to demonstrate that they learn meaningful structure–property relationships.

==============================================================


 *Targets are continuous numerical values CO₂ uptake in mol/kg,so this is a regression problem.*

- Linear regression: predicts continuous values such as 2.4 mol/kg.
- Logistic regression: predicts categories such as high/low uptake or 0/1.

In [8]:
#Train  Linear Regression Model
# Create a linear regression model, train it on  training data, 
# and make predictions on test data
#  task: Create a train the model
model = LinearRegression()
# Train the model
model.fit(X_train, y_train)
# Make predictions
linear_predictions = model.predict(X_valid)

mse = mean_squared_error(y_valid, linear_predictions)
rmse = np.sqrt(mse)

print(f"Mean Squared Error: {mse:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(linear_predictions[:5])

Mean Squared Error: 0.04
Root Mean Squared Error: 0.20
[-0.06636538  0.16500986  0.21746029  0.25112625  0.14861813]


In [ ]:
#Train Ridge regression. Scaling is necessary for Ridge.
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)
ridge_predictions = ridge_model.predict(X_valid)
print(ridge_predictions[:5])

[-0.06631551  0.16500048  0.21743688  0.25113649  0.14860263]


In [10]:
comparison = pd.DataFrame({
    "Actual": y_valid.iloc[:5],
    "Predicted_ridge": ridge_predictions[:5],
    "Predicted_linear":linear_predictions[:5]

})

print(comparison)

         Actual  Predicted_ridge  Predicted_linear
21214  0.023901        -0.066316         -0.066365
25107  0.051266         0.165000          0.165010
2383   0.167260         0.217437          0.217460
15497  0.329274         0.251136          0.251126
29148  0.066501         0.148603          0.148618


### Linear vs. Ridge Regression (0.01 bar)

Ridge and Linear Regression produced nearly identical predictions for CO2 uptake at **0.01 bar**, suggesting that Ridge regularization had minimal impact in this setting.

**Key observations:**
- Predictions include both overestimation and underestimation.
- At least one prediction is negative, which is not physically meaningful for CO2 uptake.
- This indicates that simple linear models may not fully capture adsorption behavior at very low pressure.

**Implication for next step:**
- Evaluate nonlinear models to test whether they improve predictive accuracy and physical realism.



In [11]:

# Store the validation results for each model
results_01 = []

# Evaluate the three models
models_predictions_01 = {
    "Dummy": dummy_predictions,
    "Linear Regression": linear_predictions,
    "Ridge Regression": ridge_predictions
}

for model_name_01, predictions_01 in models_predictions_01.items():

    mae = mean_absolute_error(y_valid, predictions_01)
    rmse = np.sqrt(mean_squared_error(y_valid, predictions_01))
    r2 = r2_score(y_valid, predictions_01)

    results_01.append({
        "Model": model_name_01,
        "MAE_01": mae,
        "RMSE_01": rmse,
        "R²_01": r2
    })

# Create the comparison table
comparison_table_01 = pd.DataFrame(results_01)

print(f"Validation rows: {len(y_valid)}")
print(f"Validation target mean: {y_valid.mean():.12f}")
comparison_table_01

Validation rows: 6247
Validation target mean: 0.129282020297


,Model,MAE_01,RMSE_01,R²_01
0,Dummy,0.127524,0.224469,-0.000283
1,Linear Regression,0.110546,0.198692,0.216257
2,Ridge Regression,0.110542,0.198692,0.216256


### Validation Performance Summary (0.01 bar)

The Linear and Ridge models are better than the Dummy baseline, but their predictive performance is still limited.

- **Dummy:** MAE = 0.126906, RMSE = 0.224773, R2 = -0.000313  
  It explains none of the variation in CO2 uptake.
- **Linear Regression:** MAE = 0.108584, RMSE = 0.199445, R2 = 0.212419  
    It explains about **21.2%** of the variation at 0.01 bar.
- **Ridge Regression:** MAE = 0.108577, RMSE = 0.199445, R2 = 0.212421.  
    Ridge regularization has almost no effect.

**Short interpretation:**

> Linear and Ridge Regression outperform the Dummy baseline, showing that the four structural features contain some useful information for predicting CO2 uptake at 0.01 bar. However, their R2 of approximately 0.212 means they explain only 21.2% of the variation, indicating weak predictive performance. Their nearly identical results show that Ridge regularization provides no meaningful improvement. This limited performance is reasonable at very low pressure, where CO2 uptake may depend strongly on chemical binding characteristics not included in the four structural features.

In [9]:
# We will use RandomForestRegressor as CO2 uptake are continous value 
# Random Forest Regression& Gradient Boosting Regression: Tree-based models split features using thresholds, 
# so different feature scales do not affect them.

# Create a RandonFrostRegressor 
model_rf = RandomForestRegressor (n_estimators= 100, # number of decision tree,
                                    max_depth=15, # Maximum depth of each tree,
                                    random_state=12345, # Make results reproducilbe
                                    n_jobs=1)         # uses all availavble CPU cores
# Train using the origninal , non standarized features
model_rf.fit(X_train, y_train)

# Make predictions for the training and validation sets
predictated_train_rf = model_rf.predict(X_train)
predictated_valid_rf = model_rf.predict(X_valid)

# step 2 : Calculated regression metrics
# Training metric
mae_train_rf = mean_absolute_error(y_train, predictated_train_rf)
rmse_train_rf =np.sqrt(mean_squared_error(y_train, predictated_train_rf))
r2_train_rf = r2_score(y_train, predictated_train_rf)

# validation metric
mae_valid_rf = mean_absolute_error(y_valid, predictated_valid_rf)
rmse_valid_rf = np.sqrt(mean_squared_error(y_valid, predictated_valid_rf))
r2_valid_rf = r2_score(y_valid, predictated_valid_rf)

print("=" * 45)
print("Random Forest Regression")
print("=" * 45)

print(f"Training MAE:       {mae_train_rf:.6f}")
print(f"Validation MAE:     {mae_valid_rf:.6f}")

print(f"Training RMSE:      {rmse_train_rf:.6f}")
print(f"Validation RMSE:    {rmse_valid_rf:.6f}")

print(f"Training R²:        {r2_train_rf:.6f}")
print(f"Validation R²:      {r2_valid_rf:.6f}")

print("=" * 45)

Random Forest Regression
Training MAE:       0.048660
Validation MAE:     0.080780
Training RMSE:      0.101485
Validation RMSE:    0.171085
Training R²:        0.797815
Validation R²:      0.418921


In [ ]:


# Values near  current settings
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [8, 10, 12],
    "min_samples_split": [3, 5, 8],
    "min_samples_leaf": [1, 2, 4]
}

# Create five cross-validation folds
cv_method = KFold(
    n_splits=5,
    shuffle=True,
    random_state=12345
)

# Create Grid Search using current model
grid_search_rf = GridSearchCV(
    estimator=model_rf,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv_method,
    n_jobs=-1,
    verbose=1
)

# Search using only the training data
grid_search_rf.fit(X_train, y_train)

# Display the best settings
print("Best parameters:", grid_search_rf.best_params_)
print("Best cross-validation MAE:", -grid_search_rf.best_score_)


best_model_rf_cross = grid_search_rf.best_estimator_

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters: {'max_depth': 12, 'min_samples_leaf': 4, 'min_samples_split': 3, 'n_estimators': 200}
Best cross-validation MAE: 0.07729281481595561


In [11]:
best_model_rf_cross = grid_search_rf.best_estimator_

predicted_train_rf_cross = best_model_rf_cross.predict(X_train)
predicted_valid_rf_cross = best_model_rf_cross.predict(X_valid)

mae_train_rf_cross = mean_absolute_error(
    y_train, predicted_train_rf_cross)
mae_valid_rf_cross = mean_absolute_error(
    y_valid, predicted_valid_rf_cross
)

rmse_train_rf_cross = np.sqrt(
    mean_squared_error(y_train, predicted_train_rf_cross)
)

rmse_valid_rf_cross = np.sqrt(
    mean_squared_error(y_valid, predicted_valid_rf_cross)
)

r2_train_rf_cross = r2_score(
    y_train, predicted_train_rf_cross
)

r2_valid_rf_cross = r2_score(
    y_valid, predicted_valid_rf_cross
)

print("=" * 45)
print("Cross-Validated Random Forest")
print("=" * 45)

print(f"Training MAE:       {mae_train_rf_cross:.6f}")
print(f"Validation MAE:     {mae_valid_rf_cross:.6f}")
print(f"Training RMSE:      {rmse_train_rf_cross:.6f}")
print(f"Validation RMSE:    {rmse_valid_rf_cross:.6f}")
print(f"Training R²:        {r2_train_rf_cross:.6f}")
print(f"Validation R²:      {r2_valid_rf_cross:.6f}")

print("=" * 45)

Cross-Validated Random Forest
Training MAE:       0.064092
Validation MAE:     0.079336
Training RMSE:      0.138598
Validation RMSE:    0.167335
Training R²:        0.622898
Validation R²:      0.444115


### Random Forest Performance Comparison (0.01 bar)

| Metric | Original RF | Grid-search RF | Plain-language explanation |
|---|---:|---:|---|
| Training MAE | 0.048660 | 0.064092 | Training error increased because the tuned model memorizes the training data less. |
| Validation MAE | 0.080780 | **0.079336** | On unseen MOFs, the average prediction error decreased from 0.081 to **0.079 mol/kg**. Lower is better. |
| Training RMSE | 0.101485 | 0.138598 | The increase shows that the tuned model fits the training data less aggressively. |
| Validation RMSE | 0.171085 | **0.167335** | Large prediction errors became slightly smaller. Lower is better. |
| Training R² | 0.797815 | 0.622898 | The tuned model explains less training variation because its complexity is controlled. |
| Validation R² | 0.418921 | **0.444115** | Explained variation in unseen data increased from **41.9% to 44.4%**. Higher is better. |

### Conclusion (0.01 bar)

The tuned Random Forest predicts 0.01 bar CO2 uptake with a validation MAE of 0.079 mol/kg, about 38% lower than the Dummy baseline, showing that structural descriptors provide meaningful predictive signal. With validation R² = 0.444, the model captures moderate variance and is suitable for preliminary MOF ranking and broad structure-property trend analysis, but not for high-precision uptake prediction of individual materials. Final model usefulness should be confirmed on the held-out test set and extended across other pressure targets before drawing broader conclusions.

In [12]:

# Gradient Boosting Regressor

# Create and train the baseline model
model_gb = GradientBoostingRegressor(
    random_state=12345
)

model_gb.fit(X_train, y_train)

# Training and validation predictions
predicted_train_gb = model_gb.predict(X_train)
predicted_valid_gb = model_gb.predict(X_valid)

# Training metrics
mae_train_gb = mean_absolute_error(
    y_train, predicted_train_gb)
rmse_train_gb = np.sqrt(
    mean_squared_error(y_train, predicted_train_gb))
r2_train_gb = r2_score(
    y_train, predicted_train_gb)
# Validation metrics
mae_valid_gb = mean_absolute_error(
    y_valid, predicted_valid_gb)

rmse_valid_gb = np.sqrt(
    mean_squared_error(y_valid, predicted_valid_gb))

r2_valid_gb = r2_score(
    y_valid, predicted_valid_gb)

print("=" * 45)
print("Gradient Boosting Regression")
print("=" * 45)

print(f"Training MAE:       {mae_train_gb:.6f}")
print(f"Validation MAE:     {mae_valid_gb:.6f}")
print(f"Training RMSE:      {rmse_train_gb:.6f}")
print(f"Validation RMSE:    {rmse_valid_gb:.6f}")
print(f"Training R²:        {r2_train_gb:.6f}")
print(f"Validation R²:      {r2_valid_gb:.6f}")

print("=" * 45)

Gradient Boosting Regression
Training MAE:       0.075792
Validation MAE:     0.081498
Training RMSE:      0.155751
Validation RMSE:    0.168365
Training R²:        0.523780
Validation R²:      0.437247


### Final Learning Conclusion: Gradient Boosting vs Tuned Random Forest (0.01 bar)

The Gradient Boosting model performs moderately well and shows only limited overfitting.

- Validation MAE of **0.0815 mol/kg** means predictions differ from actual uptake by about 0.082 mol/kg on average.
- Validation RMSE of **0.1684 mol/kg** indicates that some MOFs still have larger prediction errors.
- Validation R² of **0.4372** means the model explains about 43.7% of the variation in unseen MOFs.
- Training and validation metrics are relatively close, so generalization is reasonable.

### How to Read Regression Metrics

| Metric | Better Direction | Why |
|---|---|---|
| MAE | **Smaller** ↓ | Lower average prediction error |
| RMSE | **Smaller** ↓ | Lower error overall, with stronger penalty on large errors |
| R² | **Larger** ↑ | More of the target variation is explained |

### Validation Comparison

| Validation Metric | Tuned Random Forest | Gradient Boosting | Better Model |
|---|---:|---:|---|
| MAE (↓) | **0.079336** | 0.081498 | Random Forest |
| RMSE (↓) | **0.167335** | 0.168365 | Random Forest |
| R² (↑) | **0.444115** | 0.437247 | Random Forest |

For MAE specifically:

$$
0.079336 < 0.081498
$$

So Random Forest has the smaller average prediction error.

### What a Learner Should Take Away

Random Forest is currently slightly better across all validation metrics, so it is the better model **at this stage**. However, this is not the final comparison yet, because Random Forest was tuned with cross-validation while Gradient Boosting is still using default settings. A fair final decision should come after tuning Gradient Boosting with cross-validation too.

> Rule to remember: **MAE and RMSE should go down; R² should go up.**

In [13]:
# Create the baseline Gradient Boosting model
model_gb_base = GradientBoostingRegressor(
    random_state=12345
)

# Define nearby hyperparameter values
param_grid_gb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2, 4]
}

# Create five shuffled cross-validation folds
cv_method_gb = KFold(
    n_splits=5,
    shuffle=True,
    random_state=12345
)

# Create GridSearchCV
grid_search_gb = GridSearchCV(
    estimator=model_gb_base,
    param_grid=param_grid_gb,
    scoring="neg_mean_absolute_error",
    cv=cv_method_gb,
    n_jobs=-1,
    verbose=1
)

# Run the search using only training data
grid_search_gb.fit(X_train, y_train)

print("Best parameters:", grid_search_gb.best_params_)
print(
    "Best cross-validation MAE:",
    -grid_search_gb.best_score_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 4, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best cross-validation MAE: 0.07800052415189507


In [14]:
# Get the best Gradient Boosting model selected by GridSearchCV
best_model_gb = grid_search_gb.best_estimator_

# Make predictions for the training and validation datasets
predicted_train_gb_tuned = best_model_gb.predict(X_train)
predicted_valid_gb_tuned = best_model_gb.predict(X_valid)

# Calculate training metrics
mae_train_gb_tuned = mean_absolute_error(
    y_train,
    predicted_train_gb_tuned
)

rmse_train_gb_tuned = np.sqrt(
    mean_squared_error(y_train, predicted_train_gb_tuned)
)

r2_train_gb_tuned = r2_score(
    y_train,
    predicted_train_gb_tuned
)

# Calculate validation metrics
mae_valid_gb_tuned = mean_absolute_error(
    y_valid,
    predicted_valid_gb_tuned
)

rmse_valid_gb_tuned = np.sqrt(
    mean_squared_error(y_valid, predicted_valid_gb_tuned)
)

r2_valid_gb_tuned = r2_score(
    y_valid,
    predicted_valid_gb_tuned
)

# Display the results
print("=" * 45)
print("Cross-Validated Gradient Boosting")
print("=" * 45)

print(f"Training MAE:       {mae_train_gb_tuned:.6f}")
print(f"Validation MAE:     {mae_valid_gb_tuned:.6f}")

print(f"Training RMSE:      {rmse_train_gb_tuned:.6f}")
print(f"Validation RMSE:    {rmse_valid_gb_tuned:.6f}")

print(f"Training R²:        {r2_train_gb_tuned:.6f}")
print(f"Validation R²:      {r2_valid_gb_tuned:.6f}")

print("=" * 45)

Cross-Validated Gradient Boosting
Training MAE:       0.067103
Validation MAE:     0.079638
Training RMSE:      0.133122
Validation RMSE:    0.168615
Training R²:        0.652105
Validation R²:      0.435579


### Final Model Selection After Tuning (0.01 bar)

The tuned Gradient Boosting improved its validation MAE, but the tuned Random Forest still performs slightly better on all validation metrics.

| Validation metric | Tuned Random Forest | Tuned Gradient Boosting | Better |
|---|---:|---:|---|
| MAE ↓ | **0.079336** | 0.079638 | Random Forest |
| RMSE ↓ | **0.167335** | 0.168615 | Random Forest |
| R² ↑ | **0.444115** | 0.435579 | Random Forest |

For Gradient Boosting:

- MAE = **0.0796 mol/kg**: average prediction error on unseen MOFs.
- RMSE = **0.1686 mol/kg**: indicates that some predictions have larger errors.
- R² = **0.4356**: it explains approximately **43.6%** of validation uptake variation.
- Training R² of 0.6521 versus validation R² of 0.4356 indicates some overfitting.

Random Forest is the final selected model because it has lower MAE and RMSE and higher R². The difference is small, but all three metrics favor Random Forest. You can now evaluate the tuned Random Forest once on the test set.

In [15]:
# Predict the untouched test data using the selected Random Forest
predicted_test_rf = best_model_rf_cross.predict(X_test)

# Calculate final test metrics
mae_test_rf = mean_absolute_error(y_test, predicted_test_rf)

rmse_test_rf = np.sqrt(
    mean_squared_error(y_test, predicted_test_rf)
)

r2_test_rf = r2_score(y_test, predicted_test_rf)

print("=" * 45)
print("Final Random Forest Test Performance")
print("=" * 45)

print(f"Test MAE:       {mae_test_rf:.6f}")
print(f"Test RMSE:      {rmse_test_rf:.6f}")
print(f"Test R²:        {r2_test_rf:.6f}")

print("=" * 45)

Final Random Forest Test Performance
Test MAE:       0.077350
Test RMSE:      0.160441
Test R²:        0.444659


### Final Random Forest Generalization Check (0.01 bar)

| Metric | Validation | Test | Interpretation |
|---|---:|---:|---|
| MAE ↓ | 0.079336 | **0.077350** | Test average error is slightly lower. |
| RMSE ↓ | 0.167335 | **0.160441** | Large errors are slightly lower on test data. |
| R² ↑ | 0.444115 | **0.444659** | Nearly identical performance; no test-performance collapse. |

Validation and test results are very similar, indicating stable generalization.

The model is useful for preliminary screening and identifying structure-property patterns, but it is not sufficiently accurate for precise uptake prediction.

The remaining unexplained behavior may arise because the model includes only four geometric descriptors, whereas low-pressure adsorption also depends strongly on framework chemistry, functional groups, metal sites, and charge distribution.

### Final Interpretation 

Validation and test results are very similar, indicating stable generalization.

The model is useful for preliminary screening and identifying structure-property patterns, but it is not sufficiently accurate for precise uptake prediction.

The remaining unexplained behavior may arise because the model includes only four geometric descriptors, whereas low-pressure adsorption also depends strongly on framework chemistry, functional groups, metal sites, and charge distribution.

### Limitations and Future Work

#### Limitations of the Current Work

- The model explains a moderate portion of uptake variation at 0.01 bar (R² around 0.44), so predictive precision is limited for exact MOF-level uptake values.
- The feature set includes only four geometric pore descriptors, while low-pressure adsorption is also strongly influenced by chemistry-related factors.
- Current conclusions are based on one pressure target (0.01 bar), so cross-pressure behavior is not yet fully characterized.

#### Future Work

- Add chemistry-based descriptors.
- Examine residuals for high-uptake MOFs.
- Test XGBoost or LightGBM.
- Evaluate model performance across all pressures.

=======================================
## Modeling CO₂ Uptake at 0.05 bar

In [13]:
# Store the four structural features used in the relationship analysis.
feature_columns_05 = ["lcd", "pld", "void_fraction", "surface_area_m2g"]

# Store the five CO2-uptake targets in increasing pressure order.
target_05 = "CO2_uptake_0.05bar_molkg"

# Select only the required features and targets
features_05 = df[feature_columns].copy()
target_05 = df["CO2_uptake_0.05bar_molkg"].copy()
# split data
# temp simply means temporary.
# X_train: 60%
# X_temp: 40%

# Shuffle the data only when we create the train/validation/test split—before standardization and before training any model.
X_train, X_temp, y_train, y_temp = train_test_split(features_05 , target_05, test_size = 0.4 ,
                                                        random_state = 12345, shuffle=True)
                                                      
                                                       

# Train: 60%
# Validation: 20%
# Test: 20%
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=12345,
    shuffle=True
)
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_valid.shape, y_valid.shape)
print("Testing:", X_test.shape, y_test.shape)

print("\nMissing values in training features:")
print(X_train.isna().sum())

print('='*45)
# Create simple baseline model
dummy_model_05 = DummyRegressor(strategy="mean")
dummy_model_05.fit(X_train, y_train)

dummy_predictions_05 = dummy_model_05.predict(X_valid)
print(dummy_predictions_05[:2])

print('='*45)
#Train  Linear Regression Model
# Create a linear regression model, train it on  training data, 
# and make predictions on  test data
model_05 = LinearRegression()
# Train the model
model_05.fit(X_train, y_train)
# Make predictions
linear_predictions_05 = model_05.predict(X_valid)

mse = mean_squared_error(y_valid, linear_predictions_05)
rmse = np.sqrt(mse)

print(f"Mean Squared Error: {mse:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(linear_predictions_05[:5])

print('='*45)
#Train Ridge regression. Scaling is necessary for Ridge.
ridge_model_05 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge_model_05.fit(X_train, y_train)
ridge_predictions_05 = ridge_model_05.predict(X_valid)
print(ridge_predictions_05[:5])
comparison = pd.DataFrame({
    "Actual": y_valid.iloc[:5],
    "Predicted_ridge": ridge_predictions_05[:5],
    "Predicted_linear":linear_predictions_05[:5]

})

print(comparison)

print('='*45)


Training: (18740, 4) (18740,)
Validation: (6247, 4) (6247,)
Testing: (6247, 4) (6247,)

Missing values in training features:
lcd                 0
pld                 0
void_fraction       0
surface_area_m2g    0
dtype: int64
[0.43991103 0.43991103]
Mean Squared Error: 0.20
Root Mean Squared Error: 0.45
[-0.06818041  0.54405739  0.76297531  0.75872664  0.56815248]
[-0.06800507  0.54400163  0.76285504  0.75879883  0.56806116]
         Actual  Predicted_ridge  Predicted_linear
21214  0.137718        -0.068005         -0.068180
25107  0.254243         0.544002          0.544057
2383   0.987097         0.762855          0.762975
15497  1.175880         0.758799          0.758727
29148  0.350265         0.568061          0.568152


In [14]:
# Store the validation results for each model
results_05 = []

# Evaluate the three models
models_predictions_05 = {
    "Dummy": dummy_predictions_05,
    "Linear Regression": linear_predictions_05,
    "Ridge Regression": ridge_predictions_05
}

for model_name_05, predictions_05 in models_predictions_05.items():

    mae = mean_absolute_error(y_valid, predictions_05)
    rmse = np.sqrt(mean_squared_error(y_valid, predictions_05))
    r2 = r2_score(y_valid, predictions_05)

    results_05.append({
        "Model": model_name_05,
        "MAE_05": mae,
        "RMSE_05": rmse,
        "R²_05": r2
    })

# Create the comparison table
comparison_table_05 = pd.DataFrame(results_05)

print(comparison_table_05)

print('='*40)

               Model    MAE_05   RMSE_05     R²_05
0              Dummy  0.374325  0.539211 -0.000786
1  Linear Regression  0.301868  0.445586  0.316580
2   Ridge Regression  0.301860  0.445587  0.316577


In [15]:

merg_01_05 = comparison_table_01.merge(comparison_table_05, on='Model')
print(merg_01_05)


               Model    MAE_01   RMSE_01     R²_01    MAE_05   RMSE_05  \
0              Dummy  0.127524  0.224469 -0.000283  0.374325  0.539211   
1  Linear Regression  0.110546  0.198692  0.216257  0.301868  0.445586   
2   Ridge Regression  0.110542  0.198692  0.216256  0.301860  0.445587   

      R²_05  
0 -0.000786  
1  0.316580  
2  0.316577  


### Comparison of Linear Models at 0.01 and 0.05 bar

The table below compares the Dummy, Linear Regression, and Ridge Regression models on their validation sets.

| Pressure | Model | MAE (mol/kg) | RMSE (mol/kg) | R² |
|---|---|---:|---:|---:|
| 0.01 bar | Dummy | 0.127524 | 0.224469 | -0.000283 |
| 0.01 bar | Linear Regression | **0.110546** | **0.198692** | **0.216257** |
| 0.01 bar | Ridge Regression | 0.110542 | 0.198692 | 0.216256 |
| 0.05 bar | Dummy | 0.374325 | 0.539211 | -0.000786 |
| 0.05 bar | Linear Regression | **0.301868** | **0.445586** | **0.316580** |
| 0.05 bar | Ridge Regression | 0.301860 | 0.445587 | 0.316577 |

At both pressures, the Dummy model has an R² value close to zero. This is expected because it predicts the mean CO₂ uptake for every MOF and does not use the structural descriptors. It therefore explains essentially none of the variation in the validation data.

At **0.01 bar**, Linear Regression improves on the Dummy baseline: MAE decreases from 0.127524 to 0.110546 mol/kg, and RMSE decreases from 0.224469 to 0.198692 mol/kg. Its R² of 0.216257 indicates that the four structural descriptors explain approximately 21.6% of the variation in CO₂ uptake.

At **0.05 bar**, Linear Regression also outperforms the Dummy baseline. MAE decreases from 0.374325 to 0.301868 mol/kg, and RMSE decreases from 0.539211 to 0.445586 mol/kg. Its R² of 0.316580 indicates that the descriptors explain approximately 31.7% of the variation in CO₂ uptake.

Ridge Regression produces virtually the same results as Linear Regression at both pressures. The differences are negligible, so regularization provides no meaningful improvement for these targets. Linear Regression can therefore remain the main linear baseline, while Ridge need not be carried forward as a separate primary model.

The increase in R² from approximately 0.216 at 0.01 bar to 0.317 at 0.05 bar suggests that these geometric descriptors become more informative as pressure increases. This is a predictive observation, not evidence that their causal influence increases. The models still leave approximately 78.4% of the variation unexplained at 0.01 bar and 68.3% unexplained at 0.05 bar.

The remaining unexplained variation is scientifically plausible. Low-pressure adsorption can depend strongly on chemical factors that are not represented by the four descriptors, including metal identity, functional groups, electrostatic interactions, and partial charges. Simple linear models may also miss nonlinear relationships between pore structure and adsorption. The next step is therefore to evaluate nonlinear models such as Random Forest and Gradient Boosting.

### Interpreting Error Differences Across Pressure

The larger MAE and RMSE values at 0.05 bar do not by themselves indicate poorer model performance. CO₂ uptake has a larger magnitude and range at the higher pressure, so absolute errors naturally tend to be larger. MAE and RMSE are most useful for comparing models within the same pressure target. R² is more suitable for comparing the proportion of variation explained across the two targets.

In [ ]:
# We will use RandomForestRegressor as CO2 uptake are continous value 
# Random Forest Regression& Gradient Boosting Regression: Tree-based models split features using thresholds, 
# so different feature scales do not affect them.

# Create a RandonFrostRegressor 
model_rf_05 = RandomForestRegressor (n_estimators= 100, # number of decision tree,
                                    max_depth=15, # Maximum depth of each tree,
                                    random_state=12345, # Make results reproducilbe
                                    n_jobs=1)         # uses all availavble CPU cores
# Train using the origninal , non standarized features
model_rf_05.fit(X_train, y_train)

# Make predictions for the training and validation sets
predictated_train_rf_05 = model_rf_05.predict(X_train)
predictated_valid_rf_05 = model_rf_05.predict(X_valid)

# step 2 : Calculated regression metrics
# Training metric
mae_train_rf_05 = mean_absolute_error(y_train, predictated_train_rf_05)
rmse_train_rf_05 =np.sqrt(mean_squared_error(y_train, predictated_train_rf_05))
r2_train_rf_05 = r2_score(y_train, predictated_train_rf_05)

# validation metric
mae_valid_rf_05 = mean_absolute_error(y_valid, predictated_valid_rf_05)
rmse_valid_rf_05 = np.sqrt(mean_squared_error(y_valid, predictated_valid_rf_05))
r2_valid_rf_05 = r2_score(y_valid, predictated_valid_rf_05)

print("=" * 45)
print("Random Forest Regression")
print("=" * 45)

print(f"Training MAE_05:       {mae_train_rf_05:.6f}")
print(f"Validation MAE_05:     {mae_valid_rf_05:.6f}")

print(f"Training RMSE_05:      {rmse_train_rf_05:.6f}")
print(f"Validation RMSE:       {rmse_valid_rf_05:.6f}")

print(f"Training R²_05:        {r2_train_rf_05:.6f}")
print(f"Validation R²_05:      {r2_valid_rf_05:.6f}")

print("=" * 45)
print('cross-validation folds')
print("=" * 45)
# Values near  current settings
param_grid_05 = {
    "n_estimators": [100, 200],
    "max_depth": [8, 10, 12],
    "min_samples_split": [3, 5, 8],
    "min_samples_leaf": [1, 2, 4]
}

# Create five cross-validation folds
cv_method_05 = KFold(
    n_splits=5,
    shuffle=True,
    random_state=12345
)

# Create Grid Search using current model
grid_search_rf_05 = GridSearchCV(
    estimator=model_rf_05,
    param_grid=param_grid_05,
    scoring="neg_mean_absolute_error",
    cv=cv_method_05,
    n_jobs=-1,
    verbose=1
)

# Search using only the training data
grid_search_rf_05.fit(X_train, y_train)

# Display the best settings
print("Best parameters:", grid_search_rf_05.best_params_)
print("Best cross-validation MAE:", -grid_search_rf_05.best_score_)


best_model_rf_cross_05 = grid_search_rf_05.best_estimator_


Random Forest Regression
Training MAE_05:       0.129206
Validation MAE_05:     0.218353
Training RMSE_05:      0.213272
Validation RMSE:    0.366001
Training R²_05:        0.838633
Validation R²_05:      0.538906
cross-validation folds
Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters: {'max_depth': 12, 'min_samples_leaf': 4, 'min_samples_split': 3, 'n_estimators': 200}
Best cross-validation MAE: 0.2072505875400954


In [22]:
best_model_rf_cross_05 = grid_search_rf_05.best_estimator_

predicted_train_rf_cross_05 = best_model_rf_cross_05.predict(X_train)
predicted_valid_rf_cross_05 = best_model_rf_cross_05.predict(X_valid)

mae_train_rf_cross_05 = mean_absolute_error(
    y_train, predicted_train_rf_cross_05)
mae_valid_rf_cross_05 = mean_absolute_error(
    y_valid, predicted_valid_rf_cross_05
)

rmse_train_rf_cross_05 = np.sqrt(
    mean_squared_error(y_train, predicted_train_rf_cross_05)
)

rmse_valid_rf_cross_05 = np.sqrt(
    mean_squared_error(y_valid, predicted_valid_rf_cross_05)
)

r2_train_rf_cross_05 = r2_score(
    y_train, predicted_train_rf_cross_05
)

r2_valid_rf_cross_05 = r2_score(
    y_valid, predicted_valid_rf_cross_05
)

print("=" * 45)
print("Cross-Validated Random Forest")
print("=" * 45)

print(f"Training MAE_cross_05:       {mae_train_rf_cross_05:.6f}")
print(f"Validation MAE_cross_05:     {mae_valid_rf_cross_05:.6f}")
print(f"Training RMSE_cross_05:      {rmse_train_rf_cross_05:.6f}")
print(f"Validation RMSE_cross_05:    {rmse_valid_rf_cross_05:.6f}")
print(f"Training R²_cross_05:        {r2_train_rf_cross_05:.6f}")
print(f"Validation R²_cross_05:      {r2_valid_rf_cross_05:.6f}")

print("=" * 45)

Cross-Validated Random Forest
Training MAE_cross_05:       0.171538
Validation MAE_cross_05:     0.215307
Training RMSE_cross_05:      0.289844
Validation RMSE_cross_05:    0.359773
Training R²_cross_05:        0.701960
Validation R²_cross_05:      0.554466
